# Pipeline Demo — Phase 5

Demonstrates the new `portfolio.pipeline` module that replaces the
~80 lines of boilerplate duplicated across cost_analysis, validation,
risk_analysis, and alpha_iteration notebooks.

**Pipeline stages:**
1. `compute_daily_returns()` — OHLCV → daily return series
2. `compute_next_day_returns()` — daily returns → tradable next-day returns
3. `build_factor_pipeline()` — OHLCV → preprocessed composite factor
4. `build_sizing_methods()` — composite → 4 sizing method weights
5. `resample_weights()` — reduce rebalancing frequency
6. `compute_portfolio_return()` — weights × returns → portfolio returns
7. `run_alpha_pipeline()` — all-in-one wrapper

In [9]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

from data.universe import get_universe
from data.loader.data_loader import stock_load_process
from constants import DATE_COL, TICKER_COL, VALUE_COL, WEIGHT_COL, TRADING_DAYS_PER_YEAR

from portfolio.pipeline import (
    compute_daily_returns,
    compute_next_day_returns,
    compute_portfolio_return,
    resample_weights,
    build_factor_pipeline,
    build_sizing_methods,
    run_alpha_pipeline,
    list_factors,
)
from portfolio.alpha_config import AlphaConfig, FactorConfig
from portfolio.factors import get_factor_fn
from risk.position_sizing import size_equal_weight, size_half_kelly

print(f"Available factors: {list_factors()}")

Available factors: ['bbiboll', 'momentum', 'vol_ratio']


## 1. Load Data (same date range as diagnostics notebook)

In [10]:
UNIVERSE = get_universe("US_LARGE_CAP_50")
START_DATE = "2023-01-01"
END_DATE = "2026-02-28"

ohlcv = (
    stock_load_process(tickers=UNIVERSE, start_date=START_DATE, end_date=END_DATE)
    .filter(pl.col("volume") != 0)
    .collect()
)
print(f"OHLCV: {ohlcv.shape[0]:,} rows, {ohlcv.select('ticker').n_unique()} tickers")

# Precompute shared data
daily_returns = compute_daily_returns(ohlcv)
next_day_returns = compute_next_day_returns(daily_returns)
bbiboll_factor = get_factor_fn("bbiboll")(ohlcv, factor_config=FactorConfig())

print(f"Daily returns: {daily_returns.shape[0]:,} rows")
print(f"Next-day returns: {next_day_returns.shape[0]:,} rows")
print(f"BBIBOLL factor: {bbiboll_factor.shape[0]:,} rows")

Loading from cache: /mnt/blackdisk/quant_data/polygon_data/processed/us_stocks_sip/day_aggs_v1/cache_c5e11349e9e04e8bdef2634ae48ac375.parquet
Cache loaded: 40,301 rows, 2.58 MB
OHLCV: 40,093 rows, 52 tickers
Daily returns: 40,041 rows
Next-day returns: 39,989 rows
BBIBOLL factor: 38,377 rows


## 2. Ablation Study — Sizing × Rebalancing

Test all 4 combinations to isolate what flips the Sharpe from +0.814 to -0.393.

In [11]:
# Helper: compute Sharpe from weights + returns
def sharpe_from_weights(weights, ndr, rebal_n=1):
    if rebal_n > 1:
        weights = resample_weights(weights, rebal_every_n=rebal_n)
    port_ret = compute_portfolio_return(weights, ndr)
    arr = port_ret["port_return"].to_numpy()
    mu = float(np.mean(arr))
    sigma = float(np.std(arr, ddof=1))
    sharpe = mu / sigma * np.sqrt(TRADING_DAYS_PER_YEAR) if sigma > 1e-10 else 0.0
    return sharpe, mu * TRADING_DAYS_PER_YEAR, sigma * np.sqrt(TRADING_DAYS_PER_YEAR), arr, port_ret

# --- Equal-Weight weights ---
ew_weights = size_equal_weight(bbiboll_factor, n_long=10, n_short=10)

# --- Half-Kelly weights ---
returns_for_kelly = daily_returns.rename({"daily_return": "return"})
kelly_weights = size_half_kelly(
    bbiboll_factor, returns_for_kelly,
    n_long=10, n_short=10,
    lookback=60, max_position=0.10, max_leverage=1.0,
)

print(f"EW weights: {ew_weights.shape[0]:,} rows")
print(f"Kelly weights: {kelly_weights.shape[0]:,} rows")

# --- 2×2 Ablation ---
configs = {
    "A: EW + Daily":   (ew_weights, 1),
    "B: EW + Weekly":  (ew_weights, 5),
    "C: Kelly + Daily": (kelly_weights, 1),
    "D: Kelly + Weekly": (kelly_weights, 5),
}

print(f"\n{'Config':<22} {'Sharpe':>8} {'AnnRet':>8} {'AnnVol':>8}")
print("-" * 50)
ablation_results = {}
for label, (w, rebal) in configs.items():
    s, ar, av, arr, pr = sharpe_from_weights(w, next_day_returns, rebal)
    ablation_results[label] = {"sharpe": s, "arr": arr, "port_ret": pr}
    print(f"{label:<22} {s:>+8.3f} {ar:>+7.2%} {av:>7.2%}")

EW weights: 15,060 rows
Kelly weights: 14,501 rows

Config                   Sharpe   AnnRet   AnnVol
--------------------------------------------------
A: EW + Daily            +0.814  +6.86%   8.43%
B: EW + Weekly           +0.417  +3.53%   8.47%
C: Kelly + Daily         +0.137  +0.97%   7.08%
D: Kelly + Weekly        -0.393  -2.80%   7.11%


## 3. Equity Curves — Visual Comparison

In [12]:
colors = px.colors.qualitative.Set2
fig = go.Figure()
for i, (label, r) in enumerate(ablation_results.items()):
    cum = np.cumprod(1 + r["arr"])
    dates = r["port_ret"][DATE_COL].to_list()
    fig.add_trace(go.Scatter(
        x=dates, y=cum,
        name=f"{label} (S={r['sharpe']:+.2f})",
        line=dict(color=colors[i % len(colors)])
    ))
fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="BBIBOLL Ablation: Sizing × Rebalancing",
    yaxis_title="Cumulative Return",
    height=500,
)
fig.show()

## 4. Weight Diagnostics — Kelly vs Equal-Weight

Examine the weight distribution, net exposure, and concentration.

In [13]:
# --- Net exposure per date ---
def weight_stats(weights_df, label):
    stats = (
        weights_df.group_by(DATE_COL)
        .agg([
            pl.col(WEIGHT_COL).sum().alias("net_exposure"),
            pl.col(WEIGHT_COL).abs().sum().alias("gross_leverage"),
            pl.col(WEIGHT_COL).filter(pl.col(WEIGHT_COL) > 0).sum().alias("long_wt"),
            pl.col(WEIGHT_COL).filter(pl.col(WEIGHT_COL) < 0).sum().alias("short_wt"),
            pl.col(WEIGHT_COL).abs().max().alias("max_position"),
            pl.col(WEIGHT_COL).count().alias("n_positions"),
        ])
        .sort(DATE_COL)
    )
    print(f"\n--- {label} ---")
    print(f"  Dates: {stats.shape[0]}")
    print(f"  Mean net exposure:   {stats['net_exposure'].mean():+.4f}")
    print(f"  Mean gross leverage: {stats['gross_leverage'].mean():.4f}")
    print(f"  Mean long weight:    {stats['long_wt'].mean():+.4f}")
    print(f"  Mean short weight:   {stats['short_wt'].mean():+.4f}")
    print(f"  Mean max position:   {stats['max_position'].mean():.4f}")
    print(f"  Mean # positions:    {stats['n_positions'].mean():.1f}")
    return stats

ew_stats = weight_stats(ew_weights, "Equal-Weight")
kelly_stats = weight_stats(kelly_weights, "Half-Kelly")


--- Equal-Weight ---
  Dates: 753
  Mean net exposure:   -0.0000
  Mean gross leverage: 1.0000
  Mean long weight:    +0.5000
  Mean short weight:   -0.5000
  Mean max position:   0.0500
  Mean # positions:    20.0

--- Half-Kelly ---
  Dates: 726
  Mean net exposure:   -0.0073
  Mean gross leverage: 0.8928
  Mean long weight:    +0.4427
  Mean short weight:   -0.4501
  Mean max position:   0.0998
  Mean # positions:    20.0


## 5. Net Exposure Over Time — Is Kelly Creating Directional Bias?

In [14]:
# Plot net exposure over time
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ew_stats[DATE_COL].to_list(),
    y=ew_stats["net_exposure"].to_numpy(),
    name="EW net exposure",
    line=dict(color=colors[0]),
))
fig.add_trace(go.Scatter(
    x=kelly_stats[DATE_COL].to_list(),
    y=kelly_stats["net_exposure"].to_numpy(),
    name="Kelly net exposure",
    line=dict(color=colors[1]),
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Net Exposure: EW vs Kelly (0 = market-neutral)",
    yaxis_title="Net Exposure (long + short weights)",
    height=400,
)
fig.show()

## 6. Spot-Check: Who's in Each Portfolio on a Specific Date?

Pick one date and compare which stocks are long/short and their weights.

In [15]:
result = run_alpha_pipeline(
    ohlcv,
    factor_names=["bbiboll", "vol_ratio"],
    sizing_method="Half-Kelly",
    rebal_every_n=5,
    n_long=10,
    n_short=10,
)

print(f"Result keys: {sorted(result.keys())}")
print(f"\nSharpe:        {result['sharpe']:+.3f}")
print(f"Annual Return: {result['annual_return']:+.2%}")
print(f"Annual Vol:    {result['annual_vol']:.2%}")
print(f"Trading days:  {result['n_days']}")

Result keys: ['annual_return', 'annual_vol', 'composite', 'config', 'daily_returns', 'n_days', 'next_day_returns', 'portfolio_returns', 'returns_array', 'sharpe', 'sizing_methods', 'weights']

Sharpe:        -0.726
Annual Return: -4.71%
Annual Vol:    6.48%
Trading days:  725
